In [1]:
from openai import OpenAI
import pandas as pd
import sys

pd.set_option("display.max_colwidth", None)
import time
import os
from tqdm.notebook import tqdm
RANDOM_STATE = 42
N = 5000

# Connect to OpenAI
with open("./secrets.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)

In [2]:
# get labled tweets and exmples
df_labelled = pd.read_csv("output/random_sample-labelled-gpt-4o-labels.csv")
labelled_tweets_id = df_labelled['tweet_id'].tolist()
examples = df_labelled.query("label == 1").sample(n=10, random_state=RANDOM_STATE)

# new set of (unlabelled) random sample
df = (
    pd.read_parquet("gc_human_tweets.parquet")
    .reset_index(names="tweet_id")
    .query("tweet_id not in @labelled_tweets_id")
)

df_rs = (
    df
    .sample(n=N, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)
df_rs.head()

,tweet_id,username,date,author_id,content,count_hashtags,favorites,retweets,preprocessed_content,hashtags,n_hashtags,tweet_len,mentions
0,164374,thespiritdoc,2018-05-13 21:03:05,34866981,Cate Blanchett leads 82-woman Cannes red carpet ' # metoo' protest https://buff.ly/2wEdwCY,1,1,0,Cate Blanchett leads 82-woman Cannes red carpet ' #metoo' protest,[],1,65,[]
1,117988,BlackWomanHeal,2018-03-05 12:30:52,2427784531,Is the # metoo movement helping victims outside of Hollywood? Is the # OscarSunday focus on this helping others tell their stories? I’ll be on @ WKOW TONIGHT at 10:00 sharing my thoughts on this with David Johnson.pic.twitter.com/QqfXJfOeXt,2,0,0,Is the #metoo movement helping victims outside of Hollywood? Is the #OscarSunday focus on this helping others tell their stories? I’ll be on @ WKOW TONIGHT at 10:00 sharing my thoughts on this with David,[#oscarsunday],2,203,[WKOW]
2,131368,DanielleBMcLean,2018-04-26 05:47:03,1096247131,"A manager allegedly sexual harassed a subordinate at the Center for American Progress, where ThinkProgress is housed. It raises important questions about progressives and the # metoo movement https://thinkprogress.org/progressives-spaces-grappling-sexual-harassment-cded508bc1d1/ …",1,2,2,"A manager allegedly sexual harassed a subordinate at the Center for American Progress, where ThinkProgress is housed. It raises important questions about progressives and the #metoo movement …",[],1,192,[]
3,30868,naturalmaat,2018-01-19 01:58:31,25658880,# WomensMarch2018 # WomensMarchHouston # houstonwomensmarch # MeToo https://www.circlesisters13.com/blog/national-womens-march-htx …,4,1,1,#WomensMarch2018 #WomensMarchHouston #houstonwomensmarch #MeToo …,"[#womensmarch2018, #womensmarchhouston, #houstonwomensmarch]",4,65,[]
4,260076,realrezonator,2018-09-18 23:59:49,32507134,A potential danger of the # MeToo movement https://www.westernjournal.com/ct/kavanaugh-protesters-absolutely-destroyed-hilarious-cartoon/ … – bei Strands Beach,1,0,0,A potential danger of the #MeToo movement … – bei Strands Beach,[],1,63,[]


In [3]:
def build_fewshot_prompt(tweet_text, examples):
    """Build prompt with ONLY authentic examples, no not-authentic examples"""
    
    prompt = """
    Classify this #metoo tweet as "Authentic" or "Not Authentic" based on these criteria:
    
    AUTHENTIC #metoo discourse includes:
    - Personal experiences of harassment or assault (including poetry, creative writing)
    - Institutional or workplace responses to misconduct
    - Political discussions referencing #metoo (legislation, elections, speeches)
    - Meta-commentary about movement tactics or effectiveness
    - News coverage of harassment cases or accountability (including news headlines, links to articles)
    - Academic or intellectual engagement with movement themes
    - Protest or march activity where #metoo is mentioned among related causes
    - Expression of disapproval of Trump and Republican administration
    - Tweets connecting #metoo to broader social justice movements
    - Expressions of solidarity or support
    
    INAUTHENTIC tweets include:
    - Jokes
    - Sarcasm, or ironic use of #MeToo
    - Co-opting #metoo to attack Democrats
    - Expression of disapproval of Democrats
    - Dismissing or attacking survivors/accusers as liars
    
    Here are examples of AUTHENTIC tweets:
    """
    
    # Add authentic examples only
    for i, row in enumerate(examples.itertuples(), 1):
        prompt += f"{i}. \"{row.content}\"\n"
    
    prompt += f"\nNow classify this tweet:\nTweet: \"{tweet_text}\"\n\nAnswer only: Authentic or Not Authentic"
    
    return prompt


def classify_tweet(tweet_text, examples):
    """Classify with only authentic examples"""
    
    prompt = build_fewshot_prompt(tweet_text, examples)
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10,
        seed=42,
    )
    
    answer = response.choices[0].message.content.strip()
    
    if "Authentic" in answer and "Not Authentic" not in answer:
        return 1
    elif "Not Authentic" in answer:
        return 0

In [4]:
gpt_labels_auth = []
for i, (_, row) in enumerate(tqdm(df_rs.iterrows(), total=len(df_rs)), 1):
    label = classify_tweet(row["content"], examples)
    gpt_labels_auth.append(label)
    time.sleep(1.5)
    
    # Update stats
    n_auth_so_far = sum(gpt_labels_auth)
    pct_so_far = (n_auth_so_far / i) * 100
    sys.stdout.write(f"\r[{i}/{len(df_rs)}] Running: {n_auth_so_far} authentic ({pct_so_far:.1f}%)   ")
    sys.stdout.flush()

df_rs["gpt_label"] = gpt_labels_auth
df_rs.to_parquet(f"output/random_sample-{N}-fewshot-unlableled.parquet", index=False)

n_authentic = (df_rs['gpt_label'] == 1).sum()
pct_authentic = (n_authentic / len(df_rs)) * 100
print(f"\nFinal: {n_authentic} authentic ({pct_authentic:.1f}%)")

  0%|          | 0/5000 [00:00<?, ?it/s]

[5000/5000] Running: 3578 authentic (71.6%)   

Final: 3578 authentic (71.6%)
